# 1. Lightrag

- 일반 RAG
    - 텍스트 청크 저장
    - "포장불량 → 원인 → 필름장력" 검색
    - 유사도만 봄
- LightRAG
    - 엔티티 + 관계 그래프 저장
    - "포장불량 → 원인 → 필름장력" 검색
    - 관계 구조까지 봄

- 쿼리 모드:  
    - naive : 그냥 벡터 검색 (일반 RAG와 동일)  
      - QueryParam(mode="naive")

    - local : 특정 엔티티 주변 검색  
      - QueryParam(mode="local")
    
    - global : 전체 그래프 패턴 분석  
      - QueryParam(mode="global")
    
    - hybrid : local + global 동시 (가장 풍부)  
      - QueryParam(mode="hybrid")

    - Entity + Vector 검색
      - QueryParam(mode="mix")
- 비용 비교 (쿼리당 토큰):  
    - GraphRAG : 610,000 토큰  
    - LightRAG :     100 토큰(6,000배 차이)  

In [1]:
!apt-get -qq update
!apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [2]:
!pip install lightrag-hku -q
!pip install ollama -q
!pip install pdfplumber -q

In [3]:
!pkill -9 ollama

import subprocess
import time

# 백그라운드로 Ollama 서버 실행
process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
time.sleep(5)  # 서버 기동 대기
print("Ollama 서버 시작됨 (PID:", process.pid, ")")


Ollama 서버 시작됨 (PID: 10577 )


In [4]:
!ollama pull llama3:8b-instruct-q4_0
!ollama pull nomic-embed-text
!ollama pull qwen2.5:7b
!ollama pull llama3.1:8b
!ollama pull qwen2.5:3b

In [5]:
!ollama list

NAME                       ID              SIZE      MODIFIED               
qwen2.5:3b                 357c53fb659c    1.9 GB    Less than a second ago    
qwen2.5:7b                 845dbda0ea48    4.7 GB    1 second ago              
llama3.1:8b                46e0c10c039e    4.9 GB    Less than a second ago    
nomic-embed-text:latest    0a109f422b47    274 MB    1 second ago              
llama3:8b-instruct-q4_0    365c0bd3c000    4.7 GB    2 seconds ago             


In [6]:
from lightrag import LightRAG, QueryParam
from lightrag.llm.ollama import ollama_model_complete, ollama_embed
from lightrag.utils import EmbeddingFunc
import os

In [7]:
os.makedirs("./lightrag_cache", exist_ok=True)

In [8]:
!wget https://raw.githubusercontent.com/ws-l/hh_2026_5/main/data/manual.pdf

--2026-06-27 02:37:27--  https://raw.githubusercontent.com/ws-l/hh_2026_5/main/data/manual.pdf
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1432218 (1.4M) [application/octet-stream]
Saving to: ‘manual.pdf.2’

manual.pdf.2        100%[===================>]   1.37M  --.-KB/s    in 0.04s   

2026-06-27 02:37:27 (36.8 MB/s) - ‘manual.pdf.2’ saved [1432218/1432218]



In [9]:
import os, shutil, ollama
import numpy as np
from lightrag import LightRAG, QueryParam
from lightrag.llm.ollama import ollama_model_complete
from lightrag.utils import EmbeddingFunc

# 임베딩 함수 직접 구현 / numpy array 반환하도록 수정
async def custom_embed(texts):
    embeddings = []
    for text in texts:
        resp = ollama.embeddings(model="nomic-embed-text", prompt=text)
        embeddings.append(resp["embedding"])
    return np.array(embeddings, dtype=np.float32)  # numpy 변환

# 캐시 완전 삭제
shutil.rmtree("./lightrag_cache", ignore_errors=True)
os.makedirs("./lightrag_cache", exist_ok=True)

rag = LightRAG(
    working_dir="./lightrag_cache",
    llm_model_func=ollama_model_complete,
    llm_model_name="qwen2.5:3b", #"llama3.1:8b",
    llm_model_kwargs={"host": "http://localhost:11434", "timeout": 120},
    llm_model_max_async=1,        # LLM 동시 호출 1개로 제한
    embedding_func_max_async=1,   # 임베딩 동시 호출도 제한
    max_parallel_insert=1,        # 문서 동시 삽입 1개로 제한
    entity_extract_max_gleaning=0, # 재추출 시도 0회 (속도 향상)
    embedding_func=EmbeddingFunc(
        embedding_dim=768,
        max_token_size=512,
        func=custom_embed,   # 직접 구현으로 교체
    ),
)
await rag.initialize_storages()
print("초기화 완료")

INFO: [] Created new empty graph file: ./lightrag_cache/graph_chunk_entity_relation.graphml
INFO: Role LLM Configuration (initialized):
INFO:  - extract: None/None, host=None, max_async=1, timeout=240
INFO:  - keyword: None/None, host=None, max_async=1, timeout=240
INFO:  - query: None/None, host=None, max_async=1, timeout=240
INFO:  - vlm: None/None, host=None, max_async=1, timeout=240
INFO: [] Process 9625 KV load full_docs with 0 records
INFO: [] Process 9625 KV load text_chunks with 0 records
INFO: [] Process 9625 KV load full_entities with 0 records
INFO: [] Process 9625 KV load full_relations with 0 records
INFO: [] Process 9625 KV load entity_chunks with 0 records
INFO: [] Process 9625 KV load relation_chunks with 0 records
INFO: [] Process 9625 KV load llm_response_cache with 0 records
INFO: [] Process 9625 doc status load doc_status with 0 records


초기화 완료


In [10]:
await rag.ainsert("""
설비 A 베어링 과열. 원인: 윤활유 부족. 조치: 베어링 교체.
설비 B 포장 불량. 원인: 필름 장력 불균일. 조치: 파라미터 수정.
설비 C 모터 진동. 원인: 임펠러 불균형. 조치: 밸런싱 작업.
""")
print("삽입 완료")

INFO: Processing 1 document(s)
INFO: Parsing (native): doc-d850e449f0558f44959487e4996f82df
INFO: Extracting stage 1/1: unknown_source
INFO: Processing d-id: doc-d850e449f0558f44959487e4996f82df
INFO: Chunking F(legacy): size=1200, split_only=False, overlap=100, doc_id: doc-d850e449f0558f44959487e4996f82df
INFO: extract LLM func: 1 new workers initialized (Timeouts: Func: 240s, Worker: 480s, Health Check: 495s)
INFO:  == LLM cache == saving: default:extract:4cf3f9c07e8928bcdfcb518ae582ef92
INFO: Chunk 1 of 1 extracted 3 Ent + 1 Rel doc-d850e449f0558f44959487e4996f82df-chunk-000
INFO: Merging stage 1/1: unknown_source
INFO: Phase 1: Processing 3 entities from doc-d850e449f0558f44959487e4996f82df (async: 2)
INFO: Phase 2: Processing 1 relations from doc-d850e449f0558f44959487e4996f82df (async: 2)
INFO: Upserting relation VDB: `밸런싱 작업->설비 C`
INFO: Phase 2 relation processing completed for doc-d850e449f0558f44959487e4996f82df: edges=1 added_entities=1
INFO: Phase 3: Updating final 4(3+1) e

삽입 완료


In [11]:
# PDF에서 삽입할 경우
import pdfplumber

async def run_insert_pdf():
    with pdfplumber.open("manual.pdf") as pdf:
        full_text = "\n".join(
            page.extract_text() or "" for page in pdf.pages
        )
    await rag.ainsert(full_text)  #  await 추가
    print("PDF 삽입 완료")

#await run_insert_pdf()

In [12]:
!wget https://raw.githubusercontent.com/ws-l/hh_2026_5/main/data/reports.json

--2026-06-27 02:37:36--  https://raw.githubusercontent.com/ws-l/hh_2026_5/main/data/reports.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 126861 (124K) [text/plain]
Saving to: ‘reports.json.2’

reports.json.2      100%[===================>] 123.89K  --.-KB/s    in 0.01s   

2026-06-27 02:37:36 (9.44 MB/s) - ‘reports.json.2’ saved [126861/126861]



In [13]:
#await 넣어야 하는지 확인!!!!
import json, pdfplumber

# JSON: 보고서별로 분리
with open("reports.json", "r", encoding="utf-8") as f:
    reports = json.load(f)

docs = [r["report_text"] for r in reports[:20]]

# 하나씩 순차 삽입 (동시 처리 방지)
for i, doc in enumerate(docs):
    print(f"삽입 중: {i+1}/{len(docs)}")
    await rag.ainsert(doc)  # 리스트 말고 문자열 하나씩

print("완료")


INFO: Processing 1 document(s)
INFO: Parsing (native): doc-2aafbc460a1d2f564d449db5197758fc
INFO: Extracting stage 1/1: unknown_source
INFO: Processing d-id: doc-2aafbc460a1d2f564d449db5197758fc
INFO: Chunking F(legacy): size=1200, split_only=False, overlap=100, doc_id: doc-2aafbc460a1d2f564d449db5197758fc


삽입 중: 1/20


ERROR: extract LLM func: Error in decorated function for task 137650831436160_1805.202255066: 
ERROR: Failed to extract entities and relationships: C[1/1]: doc-2aafbc460a1d2f564d449db5197758fc-chunk-000: 
ERROR: Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/httpx/_transports/default.py", line 101, in map_httpcore_exceptions
    yield
  File "/usr/local/lib/python3.12/dist-packages/httpx/_transports/default.py", line 394, in handle_async_request
    resp = await self._pool.handle_async_request(req)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/httpcore/_async/connection_pool.py", line 256, in handle_async_request
    raise exc from None
  File "/usr/local/lib/python3.12/dist-packages/httpcore/_async/connection_pool.py", line 236, in handle_async_request
    response = await connection.handle_async_request(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/di

삽입 중: 2/20


INFO:  == LLM cache == saving: default:extract:9e68e5a74e72bc0f488959f2d5b47143
INFO: Chunk 1 of 1 extracted 2 Ent + 3 Rel doc-2aafbc460a1d2f564d449db5197758fc-chunk-000
INFO: Merging stage 1/2: unknown_source
INFO: Phase 1: Processing 2 entities from doc-2aafbc460a1d2f564d449db5197758fc (async: 2)
INFO: Phase 2: Processing 3 relations from doc-2aafbc460a1d2f564d449db5197758fc (async: 2)
INFO: Upserting relation VDB: `설비보전팀 검토 결과->측정결과`
INFO: Upserting relation VDB: `베어링 점검 및 교체 후 런아웃 재측정->측정결과`
INFO: Upserting relation VDB: `관련요건->측정결과`
INFO: Phase 2 relation processing completed for doc-2aafbc460a1d2f564d449db5197758fc: edges=3 added_entities=2
INFO: Phase 3: Updating final 4(2+2) entities and  3 relations from doc-2aafbc460a1d2f564d449db5197758fc
INFO: Completed merging: 2 entities, 2 extra entities, 3 relations
INFO: [] entities flush: embedding 4 vectors in 1 batch(es) (batch_num=10)
INFO: [] relationships flush: embedding 3 vectors in 1 batch(es) (batch_num=10)
INFO: [] chunks fl

...

삽입 중: 19/20


INFO:  == LLM cache == saving: default:extract:659eaa3831f3c31b323766d877a18f69
INFO: Chunk 1 of 1 extracted 3 Ent + 0 Rel doc-2c72477d2a97c4d6512cdbc494f3b628-chunk-000
INFO: Merging stage 1/1: unknown_source
INFO: Phase 1: Processing 3 entities from doc-2c72477d2a97c4d6512cdbc494f3b628 (async: 2)
INFO: Phase 2: Processing 0 relations from doc-2c72477d2a97c4d6512cdbc494f3b628 (async: 2)
INFO: Phase 3: Updating final 3(3+0) entities and  0 relations from doc-2c72477d2a97c4d6512cdbc494f3b628
INFO: Completed merging: 3 entities, 0 extra entities, 0 relations
INFO: [] entities flush: embedding 3 vectors in 1 batch(es) (batch_num=10)
INFO: [] chunks flush: embedding 1 vectors in 1 batch(es) (batch_num=10)
INFO: [] Writing graph with 139 nodes, 56 edges
INFO: In memory DB persist to disk
INFO: Completed processing file 1/1: unknown_source
INFO: Enqueued document processing pipeline stopped
INFO: Processing 1 document(s)
INFO: Parsing (native): doc-27107b86915cadc7e00692a75b922ae7
INFO: Extr

삽입 중: 20/20


INFO:  == LLM cache == saving: default:extract:b3630a89c3a38d159bc367953d219fcf
INFO: Chunk 1 of 1 extracted 3 Ent + 1 Rel doc-27107b86915cadc7e00692a75b922ae7-chunk-000
INFO: Merging stage 1/1: unknown_source
INFO: Phase 1: Processing 3 entities from doc-27107b86915cadc7e00692a75b922ae7 (async: 2)
INFO: Phase 2: Processing 1 relations from doc-27107b86915cadc7e00692a75b922ae7 (async: 2)
INFO: Upserting relation VDB: `CNC 머신 F-1호기->절삭유 압력 저하 알람 기준 설정됨`
INFO: Phase 2 relation processing completed for doc-27107b86915cadc7e00692a75b922ae7: edges=1 added_entities=1
INFO: Phase 3: Updating final 4(3+1) entities and  1 relations from doc-27107b86915cadc7e00692a75b922ae7
INFO: Completed merging: 3 entities, 1 extra entities, 1 relations
INFO: [] entities flush: embedding 4 vectors in 1 batch(es) (batch_num=10)
INFO: [] relationships flush: embedding 1 vectors in 1 batch(es) (batch_num=10)
INFO: [] chunks flush: embedding 1 vectors in 1 batch(es) (batch_num=10)
INFO: [] Writing graph with 143 

완료


In [14]:
# 4가지 쿼리 모드 비교
async def run_query():
    question = "포장 불량의 원인과 해결 방법은?"
    for mode in ["naive", "local", "global", "hybrid"]:
        print(f"\n{'='*50}\n[{mode.upper()}]")
        try:
            result = await rag.aquery(
                question,
                param=QueryParam(
                    mode=mode,
                    enable_rerank=False,
                    top_k=10,          # 40 → 10으로 줄이기
                )
            )
            print(result)
        except Exception as e:
            print(f"에러: {e}")

await run_query()


[NAIVE]


INFO: Naive query: 20 chunks (chunk_top_k:20 cosine:0.2)
INFO: Final context: 20 chunks
INFO: query LLM func: 1 new workers initialized (Timeouts: Func: 240s, Worker: 480s, Health Check: 495s)
INFO:  == LLM cache == saving: naive:query:61315925714ecd65c9458f44c0084210
INFO: keyword LLM func: 1 new workers initialized (Timeouts: Func: 240s, Worker: 480s, Health Check: 495s)


설비 B 포장 불량의 원인이 필름 장력 불균일로 판단됩니다. 이를 해결하기 위한 조치는 파라미터 수정입니다.

### References

- [1] Document Title One
- [2] Document Title Two
- [3] Document Title Three

[LOCAL]


INFO:  == LLM cache == saving: local:keywords:f1709e3abcbbe6cb4e3c88b439373414
INFO: Query edges: 포장 불량, 원인, 해결 방법 (top_k:10, cosine:0.2)
INFO: Global query: 17 entites, 10 relations
INFO: Raw search results: 17 entities, 10 relations, 0 vector chunks
INFO: After truncation: 17 entities, 10 relations
INFO: Selecting 9 from 9 entity-related chunks by vector similarity
INFO: Find no additional relations-related chunks from 10 relations
INFO: Round-robin merged chunks: 9 -> 9 (deduplicated 0)
INFO: Final context: 17 entities, 10 relations, 9 chunks
INFO: Final chunks S+F/O: E2/1 E2/2 E1/3 E3/4 E2/5 E3/6 E1/7 E2/8 E3/9
INFO:  == LLM cache == saving: local:query:9fe5edfd8e750730a3b3e1be8df7503e


2024년 1월 27일, 사출성형기 #4호기에서 확인된 포장 불량 문제는 다음과 같이 발생했습니다:

- **원인**: 필름 장력이 불균일하여 제품이 정확히 포장되지 못한 것으로 판단됩니다.
  
해당 문제가 발생한 후, 작업자들은 다음의 조치를 취하였습니다:
1. 사출성형기 #4호기에서 PC 투명 커버 표면에 나타난 흐름 자국과 미세 크랙 현상이 확인되었고, 실린더 온도가 설정 온도보다 상승하였음을 확인했습니다.
2. 설비보전팀으로부터 심각도 상으로 접수를 받아졌으며, 히터 제어 릴레이의 접점 문제가 이 문제에 관련된 것으로 추정되었습니다.
3. 이를 해결하기 위해, 히터 제어 릴레이를 교체하고 온도 측정을 다시 진행하였습니다.

결과적으로, 실린더 온도는 245±5°C의 정상 범위로 조정되었으며, 이에 따라 재성형품 검사 작업이 이루어졌습니다. 

포장 불량 문제를 해결하기 위해, 히터 제어 릴레이 교체와 온도 모니터링을 강화하는 등의 대책을 마련하였습니다.

### References

- [1] Document Title One
- [2] Document Title Two
- [3] Document Title Three

[GLOBAL]


INFO:  == LLM cache == saving: global:keywords:783c5f4ca8bca9c8ecaff21c38d32af0
INFO: Query edges: 포장, 불량, 원인, 해결 (top_k:10, cosine:0.2)
INFO: Global query: 16 entites, 10 relations
INFO: Raw search results: 16 entities, 10 relations, 0 vector chunks
INFO: After truncation: 16 entities, 10 relations
INFO: Selecting 8 from 8 entity-related chunks by vector similarity
INFO: Find no additional relations-related chunks from 10 relations
INFO: Round-robin merged chunks: 8 -> 8 (deduplicated 0)
INFO: Final context: 16 entities, 10 relations, 8 chunks
INFO: Final chunks S+F/O: E2/1 E3/2 E1/3 E3/4 E2/5 E4/6 E1/7 E2/8
INFO:  == LLM cache == saving: global:query:ee1b1164725a5407607885211c9a9e73


포장 불량 문제에 대한 원인 분석 및 해결 방법을 다음과 같이 요약할 수 있습니다:

- 원인: 필름 장력이 불균일하여 포장을 제대로 수행하지 못했습니다.
  
  보다 정확한 조치: 필름의 장력을 일정하게 유지하기 위한 파라미터를 수정하였습니다.

따라서, 포장 불량 문제는 필름의 장력 조절을 통해 해결되었습니다.

[HYBRID]


INFO:  == LLM cache == saving: hybrid:keywords:3a5dfbfa2e10a70bd5d708cfbaaab68f
INFO: Query edges: 포장, 불량, 원인, 해결 (top_k:10, cosine:0.2)
INFO: Global query: 16 entites, 10 relations
INFO: Raw search results: 16 entities, 10 relations, 0 vector chunks
INFO: After truncation: 16 entities, 10 relations
INFO: Selecting 8 from 8 entity-related chunks by vector similarity
INFO: Find no additional relations-related chunks from 10 relations
INFO: Round-robin merged chunks: 8 -> 8 (deduplicated 0)
INFO: Final context: 16 entities, 10 relations, 8 chunks
INFO: Final chunks S+F/O: E2/1 E3/2 E1/3 E3/4 E2/5 E4/6 E1/7 E2/8
INFO:  == LLM cache == saving: hybrid:query:c0b89d0afdaecd0fd582152270535f41


2024년 1월 27일 17:25, 사출성형기 #4호기에서 PC 투명 커버 표면에 흐름 자국과 미세 크랙이 동시에 발생하여 설비보전팀으로부터 심각도 상으로 접수되었습니다. 

이 문제의 원인은 파라미터 수정이 필요하다는 문서에서 언급된 필름 장력 불균일로 판단됩니다. 해결 방법은 해당 사출성형기 #4호기에서 정밀한 파라미터 설정을 변경하여 일관된 투명 커버 표면을 얻도록 조치하였습니다.

따라서, 포장 불량의 경우 필름 장력 불균일로 인해 발생하였으며 이를 수정할 수 있는 방안은 설비보전팀에서 파라미터를 재설정하는 방법입니다.


# 2. Langchain Agent tool

In [15]:
!pip install langchain_core langchain_ollama langgraph -q

In [16]:
!pkill -9 ollama

import subprocess
import time

# 백그라운드로 Ollama 서버 실행
process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
time.sleep(5)  # 서버 기동 대기
print("Ollama 서버 시작됨 (PID:", process.pid, ")")


Ollama 서버 시작됨 (PID: 13353 )


In [17]:
# rag 객체 초기화 (매번, 캐시 삭제 없이)
import numpy as np, ollama, os
from lightrag import LightRAG, QueryParam
from lightrag.llm.ollama import ollama_model_complete
from lightrag.utils import EmbeddingFunc

async def custom_embed(texts):
    embeddings = []
    for text in texts:
        resp = ollama.embeddings(model="nomic-embed-text", prompt=text)
        embeddings.append(resp["embedding"])
    return np.array(embeddings, dtype=np.float32)

# 캐시 삭제 안 함 → 기존 인덱스 재사용
rag = LightRAG(
    working_dir="./lightrag_cache",   # 기존 캐시 그대로
    llm_model_func=ollama_model_complete,
    llm_model_name="llama3.1:8b",
    llm_model_kwargs={
        "host": "http://localhost:11434",
        "timeout": 300,
        "options": {"num_predict": 256},
    },
    llm_model_max_async=1,
    embedding_func_max_async=1,
    max_parallel_insert=1,
    entity_extract_max_gleaning=0,
    embedding_func=EmbeddingFunc(
        embedding_dim=768,
        max_token_size=512,
        func=custom_embed,
    ),
)
await rag.initialize_storages()
print("rag 로드 완료 (기존 캐시 사용)")


INFO: [] Loaded graph from ./lightrag_cache/graph_chunk_entity_relation.graphml with 143 nodes, 57 edges
INFO: Role LLM Configuration (initialized):
INFO:  - extract: None/None, host=None, max_async=1, timeout=240
INFO:  - keyword: None/None, host=None, max_async=1, timeout=240
INFO:  - query: None/None, host=None, max_async=1, timeout=240
INFO:  - vlm: None/None, host=None, max_async=1, timeout=240


rag 로드 완료 (기존 캐시 사용)


In [18]:
!pip install nest_asyncio -q

import nest_asyncio
nest_asyncio.apply()  # 이벤트 루프 중첩 허용

In [19]:
import asyncio
from langchain_core.tools import tool
from lightrag import QueryParam

def run_lightrag_query(query: str, mode: str) -> str:
    """별도 이벤트 루프에서 LightRAG 쿼리 실행"""
    loop = asyncio.new_event_loop()
    try:
        return loop.run_until_complete(
            rag.aquery(query, param=QueryParam(mode=mode))
        )
    finally:
        loop.close()

@tool
def search_lightrag_local(query: str) -> str:
    """사내 설비 고장/불량 보고서에서 특정 엔티티 중심으로 검색합니다. 특정 설비명, 부품명, 불량 유형 질문에 사용하세요."""
    return run_lightrag_query(query, "local")

@tool
def search_lightrag_global(query: str) -> str:
    """사내 전체 보고서에서 패턴과 트렌드를 분석합니다. 전반적인 경향, 공통 원인 질문에 사용하세요."""
    return run_lightrag_query(query, "global")

@tool
def search_lightrag_hybrid(query: str) -> str:
    """사내 보고서에서 엔티티와 전체 패턴을 동시에 검색합니다. 복합적인 질문에 사용하세요."""
    return run_lightrag_query(query, "hybrid")

tools = [search_lightrag_local, search_lightrag_global, search_lightrag_hybrid]

In [20]:
#위에 것 안되면, 아래것으로.
import asyncio
from concurrent.futures import ThreadPoolExecutor
from langchain_core.tools import tool
from lightrag import QueryParam

# 별도 스레드 + 별도 루프에서 실행
executor = ThreadPoolExecutor(max_workers=1)

def run_lightrag_query(query: str, mode: str) -> str:
    def _run():
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            return loop.run_until_complete(
                rag.aquery(query, param=QueryParam(mode=mode))
            )
        finally:
            loop.close()
    future = executor.submit(_run)
    return future.result(timeout=120)

@tool
def search_lightrag_local(query: str) -> str:
    """사내 설비 고장/불량 보고서에서 특정 엔티티 중심으로 검색합니다. 특정 설비명, 부품명, 불량 유형 질문에 사용하세요."""
    return run_lightrag_query(query, "local")

@tool
def search_lightrag_global(query: str) -> str:
    """사내 전체 보고서에서 패턴과 트렌드를 분석합니다. 전반적인 경향, 공통 원인 질문에 사용하세요."""
    return run_lightrag_query(query, "global")

@tool
def search_lightrag_hybrid(query: str) -> str:
    """사내 보고서에서 엔티티와 전체 패턴을 동시에 검색합니다. 복합적인 질문에 사용하세요."""
    return run_lightrag_query(query, "hybrid")

tools = [search_lightrag_local, search_lightrag_global, search_lightrag_hybrid]

In [21]:
# Tool을 asyncio.run으로 단순하게
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langgraph.prebuilt import create_react_agent
from lightrag import QueryParam
import asyncio

@tool
def search_local(query: str) -> str:
    """특정 설비/부품/불량 유형 검색. 구체적인 엔티티 질문에 사용."""
    return asyncio.run(rag.aquery(query, param=QueryParam(mode="local")))

@tool
def search_global(query: str) -> str:
    """전체 보고서 패턴/트렌드 분석. 전반적인 경향 질문에 사용."""
    return asyncio.run(rag.aquery(query, param=QueryParam(mode="global")))

llm = ChatOllama(model="qwen2.5:3b", temperature=0.2, timeout=180)

agent = create_react_agent(
    model=llm,
    tools=[search_local, search_global],
    prompt="당신은 제조업 진단 전문가입니다. 반드시 tool을 사용해 답변하세요.",
)
print("Agent 준비 완료")

Agent 준비 완료


/tmp/ipykernel_9625/1231009546.py:20: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [22]:
# Agent 구성
from langchain_ollama import ChatOllama
from langgraph.prebuilt import create_react_agent

llm = ChatOllama(model="qwen2.5:3b", temperature=0.2, timeout=120)  #tool calling 지원 모형 선택

agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=(
        "당신은 제조업 엔진 진단 전문가입니다.\n"
        "- 특정 설비/부품 질문 search_lightrag_local\n"
        "- 전체 트렌드/패턴 질문 search_lightrag_global\n"
        "- 복합 질문 search_lightrag_hybrid"
    ),
    debug=True,
)


/tmp/ipykernel_9625/2844105488.py:7: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [ ]:
# 실행
questions = [
    "설비 B의 포장 불량 원인은?",
]

for q in questions:
    print(f"\n{'='*60}\n질문: {q}\n{'='*60}")
    result = agent.invoke({"messages": [{"role": "user", "content": q}]})
    print(result["messages"][-1].content)


질문: 설비 B의 포장 불량 원인은?
[values] {'messages': [HumanMessage(content='설비 B의 포장 불량 원인은?', additional_kwargs={}, response_metadata={}, id='575bcf69-1a6e-4c45-88e4-d37c6e891fe8')]}
